# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakhawat-4/my-ml-internship-v2/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
# Colab-safe: clone only if this notebook is being run standalone (not already inside the repo)
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/Sakhawat-4/my-ml-internship-v2.git"], check=True)
    os.chdir("my-ml-internship-v2")
print("Working directory ready:", os.getcwd())

Working directory ready: /home/claude/my-ml-internship-v2


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding chosen #1 — "The Freshness Multiplier" (365+ day content refreshed within the last 30
days shows a 3.2x health-score boost and 57x more impressions than similar unrefreshed old
content).**

*My methodology question:* where does the "refreshed" group come from — random assignment, or an
editor's choice? The paper compares two different populations of pages (refreshed vs
never-refreshed) rather than the same pages before and after their own refresh. If editors picked
already-promising old pages to spend refresh effort on, part of the 3.2x/57x gap is the *choosing*,
not the refresh itself — the selection-bias check `writing-honest-claims` asks for. A stronger
design would either randomize refresh timing or run a matched before/after comparison on the same
pages. This isn't a "gotcha" — the paper's own methodology page already flags it as an
observational study where "correlations do not prove causation"; my question is just applying that
caveat to this specific, most-quoted number in the paper.

**Finding chosen #2 — ML Appendix, "What Predicts Growth?" (logistic regression, 71% holdout
accuracy separating growing from declining pages, 80/20 split, 61.8K rows across 57 brands).**

*My methodology question:* what is the base rate of growing-vs-declining pages in that holdout,
and was the 80/20 split done at random row level, or grouped/time-aware across the 57 brands? The
paper doesn't state either. 71% accuracy only means something next to its base rate (per
`hunting-leakage-and-validating`'s "always print the base rate" rule) — and if the split mixed
rows from the same brand across train and test, some of that accuracy could be the model
recognizing brand-level habits rather than a portable growth signal. That's exactly the question
I'm about to turn on my own Week-5 model in Section 2, so I'd ask it of the paper first in the same
spirit: not to discredit the finding, but because the paper itself already treats the ML appendix
as secondary, exploratory evidence next to the direct aggregate comparisons.

In [2]:
# Reference table: the exact figures from the two paper findings above, transcribed from the
# report (not recomputed -- the paper's 341,701-page/57-brand portfolio is a different population
# than my 30,000-row/32-client starter dataset, so recomputing them here would not be valid).
import pandas as pd

paper_findings = pd.DataFrame([
    {"finding": "Freshness Multiplier (365+ refreshed vs not)",
     "metric": "health score", "before": 10.7, "after": 34.5, "ratio": "3.2x"},
    {"finding": "Freshness Multiplier (365+ refreshed vs not)",
     "metric": "avg impressions", "before": 71, "after": 4039, "ratio": "57x"},
    {"finding": "Growth prediction (logistic regression)",
     "metric": "holdout accuracy", "before": None, "after": 0.71, "ratio": "n=61.8K, 80/20 split"},
])
paper_findings

,finding,metric,before,after,ratio
0,Freshness Multiplier (365+ refreshed vs not),health score,10.7,34.50,3.2x
1,Freshness Multiplier (365+ refreshed vs not),avg impressions,71.0,4039.00,57x
2,Growth prediction (logistic regression),holdout accuracy,NaN,0.71,"n=61.8K, 80/20 split"


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before = a naive random row-level split** (what a first pass often looks like -- `train_test_split`
with no grouping). **After = the grouped-by-client split already used in Week 5**
(`GroupKFold`-style holdout by `client_id`, 32 clients, ~20% of clients held out entirely). Week 5
was already built honestly, so there's no separate "dishonest Week-5 result" to fix -- the useful
before/after here is the one `hunting-leakage-and-validating` itself suggests: *"report the
random-split number next to the honest-split number -- the gap between them is itself a finding
about how much memorization was happening."* Same features, same model (Random Forest, the Week-5
winner), same random seed and ~80/20 proportion -- only the split logic changes.

In [3]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"n = {len(df):,} rows | base rate (declining) = {df['is_declining_label'].mean():.3f} | "
      f"clients = {df['client_id'].nunique()}")

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c].clip(lower=0))

NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days",
           "days_since_last_update", "ctr", "avg_position", "engagement_rate",
           "scroll_rate", "ai_traffic_pct"]
CATEG = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
         "word_count_tier", "impression_tier", "position_tier"]

# Leakage guard, same as Week 5: label-derived columns and identifiers never enter the features.
excluded = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert excluded.isdisjoint(NUMERIC) and excluded.isdisjoint(CATEG)

num = df[NUMERIC].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = pd.get_dummies(df[CATEG].fillna("unknown").astype(str), prefix=CATEG)
X = pd.concat([num, cat], axis=1)
y = df["is_declining_label"]
print(f"feature matrix: {X.shape[1]} columns")

def precision_at_k(y_true, scores, k):
    d = pd.DataFrame({"y": y_true.values, "s": scores}).sort_values("s", ascending=False).head(k)
    return float(d["y"].mean()) if len(d) else 0.0

def train_and_score(Xtr, Xte, ytr, yte, label):
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                 class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(Xtr, ytr)
    p = rf.predict_proba(Xte)[:, 1]
    pred = (p >= 0.5).astype(int)
    return rf, p, {
        "split": label, "n_test": len(yte), "base_rate": yte.mean(),
        "roc_auc": roc_auc_score(yte, p), "avg_precision": average_precision_score(yte, p),
        "p@20": precision_at_k(yte, p, 20), "p@50": precision_at_k(yte, p, 50),
        "p@100": precision_at_k(yte, p, 100),
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
        "f1": f1_score(yte, pred, zero_division=0),
    }

# --- BEFORE: naive random row-level split ---
Xtr_r, Xte_r, ytr_r, yte_r, idx_tr_r, idx_te_r = train_test_split(
    X, y, df.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
rf_naive, p_naive, row_naive = train_and_score(Xtr_r, Xte_r, ytr_r, yte_r, "BEFORE: naive random split (row-level)")

train_clients_r = set(df.loc[idx_tr_r, "client_id"])
test_clients_r = set(df.loc[idx_te_r, "client_id"])
print(f"naive split: {len(train_clients_r & test_clients_r)} of {df['client_id'].nunique()} "
      f"clients appear in BOTH train and test (this is the leak)")

# --- AFTER: grouped by client_id, same design as Week 5 ---
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].unique()
shuffled_clients = rng.permutation(clients)
n_test_clients = max(1, round(len(shuffled_clients) * 0.2))
test_clients = set(shuffled_clients[:n_test_clients])
test_mask = df["client_id"].isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y.iloc[train_idx], y.iloc[test_idx]
assert set(df.iloc[train_idx]["client_id"]).isdisjoint(set(df.iloc[test_idx]["client_id"])), \
    "train/test client overlap -- split is not honest"
rf_grouped, p_grouped, row_grouped = train_and_score(Xtr_g, Xte_g, ytr_g, yte_g, "AFTER: honest split (grouped by client_id)")
print("OK: no client appears in both train and test (honest split).")

comparison = pd.DataFrame([row_naive, row_grouped]).set_index("split").round(4)
print("\n=== BEFORE / AFTER ===")
display(comparison)

gap = (comparison.loc["BEFORE: naive random split (row-level)"] - comparison.loc["AFTER: honest split (grouped by client_id)"])
print("\nGAP (naive minus honest) -- the size of this gap is itself the finding:")
print(gap.round(4))

n = 30,000 rows | base rate (declining) = 0.542 | clients = 32
feature matrix: 52 columns


naive split: 31 of 32 clients appear in BOTH train and test (this is the leak)


OK: no client appears in both train and test (honest split).

=== BEFORE / AFTER ===


,n_test,base_rate,roc_auc,avg_precision,p@20,p@50,p@100,precision,recall,f1
split,,,,,,,,,,
BEFORE: naive random split (row-level),6000,0.542,0.7579,0.7684,0.95,0.90,0.90,0.7089,0.7325,0.7205
AFTER: honest split (grouped by client_id),2325,0.391,0.7500,0.6182,0.65,0.74,0.72,0.5610,0.7437,0.6395



GAP (naive minus honest) -- the size of this gap is itself the finding:
n_test           3675.0000
base_rate           0.1510
roc_auc             0.0079
avg_precision       0.1502
p@20                0.3000
p@50                0.1600
p@100               0.1800
precision           0.1479
recall             -0.0112
f1                  0.0810
dtype: float64


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the full attack checklist from `hunting-leakage-and-validating` against the exact feature
set trained above: (1) confirm no label-derived or ID columns made it into `X` (already asserted
above, but re-checked here independently), (2) scan every feature's raw correlation with the
label and flag anything suspiciously high, (3) the harness self-test -- deliberately inject the
actual leak (`trend_pct`, the column the label is computed from) and confirm the score breaks
toward a near-perfect 1.0. If it didn't, the test harness itself would be the thing that's broken,
not the model.

In [4]:
# 1) leakage guard re-check, independent of the assert already in Section 2
excluded = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert excluded.isdisjoint(set(X.columns)), "a label-derived or ID column leaked into the features"
print("PASS: no label-derived or ID columns in the final feature set.")

# 2) correlation scan -- flag anything that looks "too good"
corrs = X.apply(lambda col: np.corrcoef(col.astype(float), y)[0, 1])
top_corrs = corrs.abs().sort_values(ascending=False).head(8)
print("\nTop 8 |correlation| with the label, across all 52 features:")
print(top_corrs.round(3))
print(f"\nHighest single-feature correlation is {top_corrs.iloc[0]:.3f} -- nowhere near the "
      f"'one feature towers over all others' pattern the skill calls a label-derived-feature symptom.")

# 3) the harness self-test: inject the KNOWN leak and confirm the test would catch it
X_leaky = X.copy()
X_leaky["LEAK_trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)
Xtr_leaky, Xte_leaky = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]

rf_leak_check = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
rf_leak_check.fit(Xtr_leaky, ytr_g)
p_leak = rf_leak_check.predict_proba(Xte_leaky)[:, 1]
auc_leak = roc_auc_score(yte_g, p_leak)
auc_honest = row_grouped["roc_auc"]

print(f"\nROC-AUC WITHOUT the leak (my real feature set): {auc_honest:.4f}")
print(f"ROC-AUC WITH the leak injected on purpose:        {auc_leak:.4f}")
verdict = "PASS -- harness correctly detects leakage" if auc_leak > 0.97 else "FAIL -- harness may be broken, investigate"
print(f"Harness self-test: {verdict}")

PASS: no label-derived or ID columns in the final feature set.

Top 8 |correlation| with the label, across all 52 features:
days_with_impressions          0.190
log_impressions_90d            0.177
position_tier_top_3            0.175
content_age_days               0.164
content_type_feedly article    0.140
impression_tier_low            0.137
age_tier_91-180                0.135
competition_level_unknown      0.135
dtype: float64

Highest single-feature correlation is 0.190 -- nowhere near the 'one feature towers over all others' pattern the skill calls a label-derived-feature symptom.



ROC-AUC WITHOUT the leak (my real feature set): 0.7500
ROC-AUC WITH the leak injected on purpose:        1.0000
Harness self-test: PASS -- harness correctly detects leakage


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest Week-5 sentence was: *"Random Forest is the clear winner on every metric above."* Two
problems with it now that I've audited it: (1) it doesn't say against what split -- Section 2 shows
that same sentence would have been badly overstated if I'd measured it on the naive random split
instead of the honest one I actually used; (2) "clear winner" reads like a settled, general fact,
when it's really a comparison observed on one 30,000-row, 32-client starter sample, on one 20%
client holdout.

In [5]:
# Ground the rewrite in the numbers actually computed above, not typed by hand.

# What an UNSAFE claim would have said if it had been written from the naive split (Section 2, "BEFORE"):
unsafe_claim = (
    f"The model correctly flags {row_naive['p@20']:.0%} of the top 20 highest-risk pages "
    f"and is the clear winner on every metric."
)

# The SAFE version, from the honest grouped split (Section 2, "AFTER") -- same model, same K:
safe_claim = (
    f"On a held-out set of client accounts the model never trained on, the Random Forest ranking "
    f"showed p@20 of {row_grouped['p@20']:.0%} (base rate {row_grouped['base_rate']:.0%}) -- "
    f"an observed, directional improvement over the Week-4 rule baseline on this same honest split, "
    f"but meaningfully lower than the naive-split number ({row_naive['p@20']:.0%}) would have "
    f"suggested. Whether this generalizes beyond these 32 clients is not yet measured."
)

print("UNSAFE (would-have-been) claim:\n ", unsafe_claim)
print("\nSAFE (rewritten) claim:\n ", safe_claim)

# Baseline vs Random Forest, both scored on the SAME honest test rows, for the decision-support framing above
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
tier_median_ctr = visible.groupby("position_tier")["ctr"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)
refresh_cond = (df["freshness_tier"] == "91-180") & (df["impressions_90d"] >= 500)
ctr_fix_cond = (df["ctr"] < df["tier_median_ctr"]) & (df["impressions_90d"] >= 100) & (df["avg_position"] > 0)
quick_win_cond = (df["impression_tier"].isin(["good", "excellent"])) & (df["position_tier"] == "striking")
ctr_gap = ((df["tier_median_ctr"] - df["ctr"]) / df["tier_median_ctr"].replace(0, np.nan)).clip(lower=0).fillna(0)
baseline_score = np.select([refresh_cond, ctr_fix_cond, quick_win_cond],
                            [df["impressions_90d"], df["impressions_90d"] * ctr_gap, df["impressions_90d"] / df["avg_position"]],
                            default=0.0)
baseline_test = baseline_score[test_idx]
print("\nOn the honest split -- baseline rule vs Random Forest (decision-support comparison):")
for k in [20, 50, 100]:
    print(f"  p@{k}: baseline={precision_at_k(yte_g, baseline_test, k):.2f}  RF={precision_at_k(yte_g, p_grouped, k):.2f}")

UNSAFE (would-have-been) claim:
  The model correctly flags 95% of the top 20 highest-risk pages and is the clear winner on every metric.

SAFE (rewritten) claim:
  On a held-out set of client accounts the model never trained on, the Random Forest ranking showed p@20 of 65% (base rate 39%) -- an observed, directional improvement over the Week-4 rule baseline on this same honest split, but meaningfully lower than the naive-split number (95%) would have suggested. Whether this generalizes beyond these 32 clients is not yet measured.

On the honest split -- baseline rule vs Random Forest (decision-support comparison):
  p@20: baseline=0.30  RF=0.65
  p@50: baseline=0.36  RF=0.74
  p@100: baseline=0.56  RF=0.72


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.